In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

In [2]:
#Check per-attack-type recall
cic_full = pd.read_csv(("../data/processed/cicids2017_wednesday_clean.csv"))
X = pd.read_csv("../data/processed/cicids2017_features_reduced.csv")
y = pd.read_csv("../data/processed/cicids2017_labels.csv").squeeze()
label_multiclass = cic_full['Label']

X_train, X_test, y_train, y_test, label_train, label_test = train_test_split(X, y, label_multiclass, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

xgb = joblib.load("../models/xgboost.pkl")

In [3]:
y_pred = xgb.predict(X_test_scaled)

results_by_type = pd.DataFrame({'true_label': label_test, 'predicted': y_pred})
results_by_type['type_binary'] = results_by_type['true_label'].apply(lambda x: 0 if x == 'BENIGN' else 1)

for attack_type in results_by_type['true_label'].unique():
    subset = results_by_type[results_by_type['true_label'] == attack_type]
    if attack_type == 'BENIGN':
        continue
    caught = (subset['predicted'] == 1).sum()
    total = len(subset)
    print(f"{attack_type}: {caught}/{total} caught ({caught/ total*100: .1f}% recall)")


DoS Hulk: 34605/34606 caught ( 100.0% recall)
DoS Slowhttptest: 1076/1077 caught ( 99.9% recall)
DoS GoldenEye: 2020/2020 caught ( 100.0% recall)
DoS slowloris: 1045/1047 caught ( 99.8% recall)
Heartbleed: 1/1 caught ( 100.0% recall)


In [4]:
# Apply SMOTE and compare

from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print("Before SMOTE:", y_train_smote.value_counts().to_dict())
print("After AMOTE:", pd.Series(y_train_smote).value_counts().to_dict())

Before SMOTE: {0: 333388, 1: 333388}
After AMOTE: {0: 333388, 1: 333388}


In [ ]:
from xgboost import XGBClassifier

xgb_smote = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, random_state=42)
xgb_smote.fit(X_train_smote, y_train_smote)

y_pred_smote = xgb_smote.predict(X_test_scaled)
results_by_type['predicted_smote'] = y_pred_smote

print("\n--- After SMOTE ---")
for attack_type in results_by_type['true_label'].unique():
    subset = results_by_type[results_by_type['true_label'] == attack_type]
    if attack_type == 'BENIGN':
        continue
    caught = (subset['predicted_smote'] == 1).sum()
    total = len(subset)
    print(f"{attack_type}: {caught}/{total} caught ({caught/ total*100: .1f}% recall)")



--- After SMOTE ---
DoS Hulk: 34605/34606 caught ( 100.0% recall)
DoS Slowhttptest: 1076/1077 caught ( 99.9% recall)
DoS GoldenEye: 2020/2020 caught ( 100.0% recall)
DoS slowloris: 1043/1047 caught ( 99.6% recall)
Heartbleed: 1/1 caught ( 100.0% recall)


In [6]:
# Compare overall metrices too

from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

print("Before SMOTE- Prediction:", precision_score(y_test, y_pred), "Recall:", recall_score(y_test, y_pred), "F1:", f1_score(y_test, y_pred))
print("After SMOTE- Prediction:", precision_score(y_test, y_pred_smote), "Recall:", recall_score(y_test, y_pred_smote), "F1:", f1_score(y_test, y_pred_smote))

joblib.dump(xgb_smote, "../models/xgboost_smote.pkl")

Before SMOTE- Prediction: 0.9996388122081473 Recall: 0.9998967768573714 F1: 0.9997677778924554
After SMOTE- Prediction: 0.9994325070291743 Recall: 0.9998451652860572 F1: 0.9996387935705255


['../models/xgboost_smote.pkl']

In [7]:
import json

sample_row = X_test.iloc[0].to_dict()
print(json.dumps({"features": sample_row}, indent=2))

{
  "features": {
    "Destination Port": 53.0,
    "Flow Duration": 59813.0,
    "Total Fwd Packets": 4.0,
    "Total Length of Fwd Packets": 96.0,
    "Fwd Packet Length Max": 24.0,
    "Fwd Packet Length Min": 24.0,
    "Fwd Packet Length Mean": 24.0,
    "Bwd Packet Length Max": 56.0,
    "Bwd Packet Length Min": 56.0,
    "Flow Bytes/s": 5350.007523,
    "Flow Packets/s": 133.7501881,
    "Flow IAT Mean": 8544.714286,
    "Flow IAT Std": 14520.58165,
    "Flow IAT Min": 1.0,
    "Fwd IAT Min": 1.0,
    "Bwd IAT Total": 29065.0,
    "Bwd IAT Mean": 9688.333333,
    "Bwd IAT Std": 16735.66671,
    "Bwd IAT Min": 4.0,
    "Fwd PSH Flags": 0.0,
    "Bwd PSH Flags": 0.0,
    "Fwd URG Flags": 0.0,
    "Bwd URG Flags": 0.0,
    "Bwd Packets/s": 66.87509404,
    "Min Packet Length": 24.0,
    "FIN Flag Count": 0.0,
    "RST Flag Count": 0.0,
    "PSH Flag Count": 0.0,
    "ACK Flag Count": 0.0,
    "URG Flag Count": 0.0,
    "CWE Flag Count": 0.0,
    "Down/Up Ratio": 1.0,
    "Fwd Avg By